In [0]:
df = spark.table("cafe_sales_2026.silver.clean_cafe_sales")
display(df)

In [0]:
from pyspark.sql.functions import expr, col

# Tabela gold no grão de transação (uma linha = uma venda)
# Serve como fonte única de verdade para análises ad-hoc e para as agregações
df_fact = (
    df
    .withColumn("Transaction_Date_Parsed", expr("try_to_date(Transaction_Date, 'yyyy-MM-dd')"))
    .select(
        col("Transaction_ID"),
        col("Item"),
        col("Quantity"),
        col("Price_Per_Unit"),
        col("Total_Spent"),
        col("Payment_Method"),
        col("Location"),
        col("Transaction_Date_Parsed").alias("Transaction_Date")
    )
)

display(df_fact)

In [0]:
df_fact.write.mode("overwrite").saveAsTable("cafe_sales_2026.gold.fact_cafe_sales")

In [0]:
from pyspark.sql.functions import sum as spark_sum, col

df_gold_item = (
    df.groupBy("Item")
      .agg(
          spark_sum("Quantity").alias("Total_Quantity"),
          spark_sum("Total_Spent").alias("Total_Revenue")
      )
      .orderBy(col("Total_Revenue").desc())
)

display(df_gold_item)

In [0]:
df_gold_item.write.mode("overwrite").saveAsTable("cafe_sales_2026.gold.sales_by_item")

In [0]:
df_gold_payment = (
    df.groupBy("Payment_Method")
      .agg(
          spark_sum("Quantity").alias("Total_Quantity"),
          spark_sum("Total_Spent").alias("Total_Revenue")
      )
      .orderBy(col("Total_Revenue").desc())
)

display(df_gold_payment)

In [0]:
df_gold_payment.write.mode("overwrite").saveAsTable("cafe_sales_2026.gold.sales_by_payment_method")

In [0]:
df_gold_location = (
    df.groupBy("Location")
      .agg(
          spark_sum("Quantity").alias("Total_Quantity"),
          spark_sum("Total_Spent").alias("Total_Revenue")
      )
      .orderBy(col("Total_Revenue").desc())
)

display(df_gold_location)

In [0]:
df_gold_location.write.mode("overwrite").saveAsTable("cafe_sales_2026.gold.sales_by_location")

In [0]:
df.select("Transaction_Date").distinct().limit(5).display()

In [0]:
from pyspark.sql.functions import expr, col

df_gold_date = (
    df
    .withColumn("Transaction_Date_Parsed", expr("try_to_date(Transaction_Date, 'yyyy-MM-dd')"))
    .filter(col("Transaction_Date_Parsed").isNotNull())  # exclui só nesta agregação, silver continua intacta
    .groupBy("Transaction_Date_Parsed")
    .agg(
        spark_sum("Quantity").alias("Total_Quantity"),
        spark_sum("Total_Spent").alias("Total_Revenue")
    )
    .withColumnRenamed("Transaction_Date_Parsed", "Transaction_Date")
    .orderBy("Transaction_Date")
)

display(df_gold_date)

In [0]:
df_gold_date.write.mode("overwrite").saveAsTable("cafe_sales_2026.gold.sales_by_date")